# Notebook 4 — Feature Engineering y Selección de Variables 🎯
**TFG: Detección de Intrusiones a Gran Escala utilizando ML Distribuido y Big Data**

**Autor:** Eduardo Morillas Rodríguez

**Entrada:** Dataset limpio de NB3 (`DATA_CLEAN_PATH/preprocessed`)
**Salida:** Dataset con features seleccionadas (`DATA_FINAL_PATH`)

---

## Relación con notebooks anteriores

| NB | Qué aporta a NB4 |
|----|-------------------|
| NB1 | Auditoría: detectó infinitos, duplicados, negativos (ya limpiados en NB3) |
| NB2 | EDA: heatmap de correlación (vis 10, 14) → bloques redundantes; patrones temporales (vis 20-22) |
| NB3 | Dataset limpio con features individuales + Label + Timestamp |

## Qué hace este notebook

✅ Extrae features temporales desde `Timestamp` (hora, día semana)

✅ Elimina features redundantes por alta correlación

✅ Analiza importancia de features (RF, PCA, ANOVA F-test, VIF)

✅ Guarda dataset con features seleccionadas para NB5



## Qué NO hace (y por qué)

| Operación | ¿Dónde? | Motivo |
|-----------|---------|--------|
| Balanceo de clases | NB5 | Debe hacerse DESPUÉS del train/test split para evitar data leakage |
| VectorAssembler | NB5 | Depende del conjunto final de features post-selección |
| StandardScaler | NB5 | Fit solo en train para evitar data leakage |
| Label encoding final | NB5 | Se crea aquí temporalmente para análisis (RF, ANOVA) |

## Respaldo en la literatura

- **Göcs & Johanyák (2023)**: Aplican 6 métodos de feature selection al CSE-CIC-IDS 2018;
  eliminan features redundantes por correlación antes de clasificación.
- **Songma et al. (2023, MDPI Computers)**: Combinan PCA + RF feature importance
  para reducir dimensionalidad; XGBoost, DT y RF logran mejores ROC.
- **Zouhri et al. (2024, Int. J. Inf. Security)**: Evalúan 5 filtros univariantes
  (Relieff, Pearson, MI, ANOVA, Chi²) + 3 multivariantes sobre CSE-CIC-IDS 2018;
  XGBoost y RF con filtros multivariantes mantienen rendimiento con menos features.
- **Parlanti & Catania (2025, arXiv)**: Framework temporal para IDS;
  features temporales capturan patrones de ataque periódicos.
- **Shanmugam et al. (2025, MDPI Electronics)**: Evalúan resampling para IDS;
  el balanceo debe hacerse SOLO sobre train para evitar data leakage.

---
## Imports y Configuración

In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType
from pyspark.ml.feature import (
	StringIndexer, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

from config import *

spark = get_spark_session("TFG_IDS_NB4_FeatureEng")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 22:56:50 WARN Utils: Your hostname, eespcachcpro01, resolves to a loopback address: 127.0.1.1; using 10.151.52.2 instead (on interface ens3f0)
26/05/04 22:56:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 22:56:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/04 22:56:50 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


---
## 4.1 — Carga del Dataset Limpio (NB3)

In [2]:
df = spark.read.parquet(os.path.join(DATA_CLEAN_PATH, "preprocessed"))
initial_rows = df.count()
initial_cols = len(df.columns)

print(f"📊 Dataset cargado desde NB3:")
print(f"  Filas:	{initial_rows:,}")
print(f"  Columnas: {initial_cols}")
print(f"\nColumnas disponibles:\n{df.columns}")

[Stage 1:====================================>                  (84 + 44) / 128]

📊 Dataset cargado desde NB3:
  Filas:	15,705,438
  Columnas: 72

Columnas disponibles:
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow F

---
## 4.2 — Feature Engineering Temporal (desde `Timestamp`)

**Motivación (NB2 — EDA):**
- Vis 20: Volumen de tráfico con patrones claros por hora del día
- Vis 21a/b: Distribución temporal de clases de ataque
- Vis 22: Heatmap hora × día de la semana muestra concentraciones

**Literatura:**
- Parlanti & Catania (2025): Las features temporales capturan patrones
periódicos de ataques que las features de flujo no recogen.

Se derivan:
- `hour` (0-23)
- `day_of_week` (1=domingo, 7=sábado)

Tras la extracción, `Timestamp` se elimina (ya no aporta).

In [3]:
print("🔧 Derivando features temporales desde Timestamp...")

if "Timestamp" in df.columns:
	df = df.withColumn(
    	"_ts",
    	F.to_timestamp(F.col("Timestamp"), "dd/MM/yyyy HH:mm:ss")
	)

	df = (
    	df
    	.withColumn("hour", F.hour("_ts"))
    	.withColumn("day_of_week", F.dayofweek("_ts"))
	)

	# Verificación
	n_null_ts = df.filter(F.col("hour").isNull()).count()
	print(f"  ✅ Features temporales creadas: hour, day_of_week")
	print(f"  ℹ️ is_weekend no se crea: el dataset CSE-CIC-IDS 2018 fue capturado")
	print(f" 	exclusivamente en días laborables (14-23 feb 2018, lun-vie)")
	print(f"  Nulos en features temporales: {n_null_ts}")

	# Eliminar Timestamp y columna auxiliar
	df = df.drop("Timestamp", "_ts")
	print(f"  ✅ Timestamp eliminado tras feature engineering")
else:
	print("  ⚠️ Columna 'Timestamp' no encontrada — se omite")

print(f"  Columnas actuales: {len(df.columns)}")

🔧 Derivando features temporales desde Timestamp...


[Stage 4:=============================================>        (109 + 19) / 128]

  ✅ Features temporales creadas: hour, day_of_week
  ℹ️ is_weekend no se crea: el dataset CSE-CIC-IDS 2018 fue capturado
 	exclusivamente en días laborables (14-23 feb 2018, lun-vie)
  Nulos en features temporales: 0
  ✅ Timestamp eliminado tras feature engineering
  Columnas actuales: 73


---
## 4.3 — Preparación: Label Encoding (para análisis)

Se crea `label_index` necesario para los métodos de selección de features
(RF Feature Importance, ANOVA F-test). Este encoding se mantiene en el
dataset guardado por conveniencia, pero NB5 puede recrearlo si es necesario.

In [4]:
print("🔧 Encoding de Label → label_index (para análisis)...")

label_indexer = StringIndexer(
	inputCol="Label",
	outputCol="label_index",
	handleInvalid="error"
)
label_model = label_indexer.fit(df)
df = label_model.transform(df)

print(f"\n📋 Mapeo de labels:")
for idx, label in enumerate(label_model.labels):
	print(f"  {idx:<3} → {label}")

🔧 Encoding de Label → label_index (para análisis)...



📋 Mapeo de labels:
  0   → Benign
  1   → DDoS attack-HOIC
  2   → DDoS attacks-LOIC-HTTP
  3   → DoS attacks-Hulk
  4   → Bot
  5   → Infilteration
  6   → SSH-Bruteforce
  7   → DoS attacks-GoldenEye
  8   → FTP-BruteForce
  9   → DoS attacks-SlowHTTPTest
  10  → DoS attacks-Slowloris
  11  → DDoS attack-LOIC-UDP
  12  → Brute Force -Web
  13  → Brute Force -XSS
  14  → SQL Injection


In [5]:
# Definir feature_cols (solo numéricas, excluyendo Label y label_index)
exclude_cols = {"Label", "label_index"}
feature_cols = [
	f.name for f in df.schema.fields
	if f.name not in exclude_cols
	and isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]
print(f"\n📊 Features numéricas para análisis: {len(feature_cols)}")


📊 Features numéricas para análisis: 72


---
## 4.4 — Eliminación por Alta Correlación (> 0.95)

**Motivación (NB2 — EDA):**
- Vis 10 (heatmap de correlación): bloques de features casi idénticas
- Vis 14 (clustermap): grupos jerárquicos de features redundantes
- Vis A1 (scatter alta correlación): relaciones lineales ≈ perfectas

**Literatura:**
- Göcs & Johanyák (2023): eliminan features redundantes por correlación
  antes de clasificación, manteniendo accuracy.
- Zouhri et al. (2024): Pearson correlation como filtro univariante
  reduce dimensionalidad sin pérdida significativa de rendimiento.

**Umbral:** `CORRELATION_THRESHOLD = 0.95` (definido en config.py)

In [6]:
print(f"🔍 Calculando correlación (umbral > {CORRELATION_THRESHOLD})...")

SAMPLE_CORR = min(100_000, initial_rows)
df_corr_pd = (
    df.select(feature_cols)
    .sample(False, SAMPLE_CORR / initial_rows, seed=SEED)
    .toPandas()
)
df_corr_pd = df_corr_pd.apply(pd.to_numeric, errors="coerce")
corr_matrix = df_corr_pd.corr(method="pearson").abs()

# Triángulo superior
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

cols_to_remove_corr = set()
high_corr_pairs = []
for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if pd.notna(val) and val > CORRELATION_THRESHOLD:
            high_corr_pairs.append((idx, col, round(val, 4)))
            cols_to_remove_corr.add(col)

print(f"  Pares con correlación > {CORRELATION_THRESHOLD}: {len(high_corr_pairs)}")
print(f"  Features a eliminar: {len(cols_to_remove_corr)}")

# Mostrar los primeros pares
for a, b, v in sorted(high_corr_pairs, key=lambda x: -x[2])[:10]:
    print(f"	· {a} ↔ {b}: {v}")

feature_cols_reduced = [c for c in feature_cols if c not in cols_to_remove_corr]
print(f"\n  Features: {len(feature_cols)} → {len(feature_cols_reduced)}")

🔍 Calculando correlación (umbral > 0.95)...


26/05/04 22:57:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  Pares con correlación > 0.95: 48
  Features a eliminar: 27
	· Fwd PSH Flags ↔ SYN Flag Cnt: 1.0
	· Fwd URG Flags ↔ CWE Flag Count: 1.0
	· Fwd Pkt Len Mean ↔ Fwd Seg Size Avg: 1.0
	· Bwd Pkt Len Mean ↔ Bwd Seg Size Avg: 1.0
	· Tot Fwd Pkts ↔ Subflow Fwd Pkts: 1.0
	· TotLen Fwd Pkts ↔ Subflow Fwd Byts: 1.0
	· Tot Bwd Pkts ↔ Subflow Bwd Pkts: 1.0
	· TotLen Bwd Pkts ↔ Subflow Bwd Byts: 1.0
	· Tot Bwd Pkts ↔ Bwd Header Len: 0.9999
	· RST Flag Cnt ↔ ECE Flag Cnt: 0.9999

  Features: 72 → 45


---
## 4.5 — Feature Importance (Random Forest Ligero)

**Literatura:**
- Songma et al. (2023): Combinan PCA + RF feature importance;
  RF identifica features discriminantes para cada tipo de ataque.
- Göcs & Johanyák (2023): Usan 6 métodos de selección incluyendo RF importance.

**Nota:** Se entrena un RF ligero (30 árboles, maxDepth=10) sobre una muestra
del 30%. Es un análisis exploratorio, NO el modelo final.

In [7]:
print("🔍 Feature Importance con RF ligero...")

assembler_fi = VectorAssembler(
    inputCols=feature_cols_reduced,
    outputCol="features_fi",
    handleInvalid="skip"
)
rf_fi = RandomForestClassifier(
    labelCol="label_index",
    featuresCol="features_fi",
    numTrees=30,
    maxDepth=10,
    seed=SEED
)
pipeline_fi = Pipeline(stages=[assembler_fi, rf_fi])

model_fi = pipeline_fi.fit(
    df.sample(False, min(0.3, 1.0), seed=SEED)
)
importances = model_fi.stages[1].featureImportances.toArray()
importance_df = (
    pd.DataFrame({"feature": feature_cols_reduced, "importance": importances})
    .sort_values("importance", ascending=False)
)

print("\n📊 Top 20 features por importancia (Gini):")
print(importance_df.head(20).to_string(index=False))

# Visualización
fig, ax = plt.subplots(figsize=(14, 10))
top20 = importance_df.head(20)
ax.barh(top20["feature"][::-1], top20["importance"][::-1], color="#3498db")
ax.set_xlabel("Importancia (Gini)")
ax.set_title("Feature Importance — Random Forest (Top 20)")
plt.tight_layout()
save_figure(fig, "31_feature_importance_rf")

🔍 Feature Importance con RF ligero...


26/05/04 22:57:41 WARN DAGScheduler: Broadcasting large task binary with size 1187.3 KiB
26/05/04 22:57:43 WARN DAGScheduler: Broadcasting large task binary with size 2016.7 KiB
26/05/04 22:57:46 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/05/04 22:57:50 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
                                                                                


📊 Top 20 features por importancia (Gini):
          feature  importance
Init Fwd Win Byts    0.131482
         Dst Port    0.092430
             hour    0.067715
 Fwd Seg Size Min    0.059388
     Flow IAT Max    0.058089
      day_of_week    0.049859
    Flow IAT Mean    0.048712
  Fwd Pkt Len Max    0.045887
      Flow Pkts/s    0.045869
 Fwd Pkt Len Mean    0.045563
    Flow Duration    0.043752
       Bwd Pkts/s    0.025871
     Tot Bwd Pkts    0.025061
      Flow Byts/s    0.024052
 Bwd Pkt Len Mean    0.023271
     Tot Fwd Pkts    0.022231
      Fwd IAT Std    0.021725
        Idle Mean    0.020510
  Bwd Pkt Len Max    0.019492
     Pkt Len Mean    0.018602
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/31_feature_importance_rf.png


---
## 4.6 — PCA (Spark MLlib)

**Literatura:**
- Songma et al. (2023): PCA como método complementario a RF
  para reducción de dimensionalidad.

**Nota:** El StandardScaler aplicado aquí es SOLO para el análisis PCA
(PCA requiere datos estandarizados). NO se usa para el dataset final —
el escalado definitivo se hará en NB5 tras train/test split.

In [8]:
print("🔍 PCA con Spark MLlib...")

from pyspark.ml.feature import PCA as SparkPCA

assembler_pca = VectorAssembler(
    inputCols=feature_cols_reduced,
    outputCol="features_pca_raw",
    handleInvalid="skip"
)
scaler_pca = StandardScaler(
    inputCol="features_pca_raw",
    outputCol="features_pca_scaled",
    withMean=True,
    withStd=True
)
pca_full = SparkPCA(
    k=len(feature_cols_reduced),
    inputCol="features_pca_scaled",
    outputCol="pca_features_full"
)

pca_model = Pipeline(stages=[assembler_pca, scaler_pca, pca_full]).fit(df)
explained_variance = pca_model.stages[2].explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)
n_components_95 = int(np.argmax(cumulative_variance >= PCA_VARIANCE_THRESHOLD) + 1)

print(f"  {len(feature_cols_reduced)} features → {n_components_95} componentes ({PCA_VARIANCE_THRESHOLD*100:.0f}% varianza)")

# Visualización
fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(range(1, len(explained_variance)+1), explained_variance,
    alpha=0.6, color="#3498db", label="Individual")
ax.plot(range(1, len(cumulative_variance)+1), cumulative_variance,
        "r-o", markersize=4, label="Acumulada")
ax.axhline(y=PCA_VARIANCE_THRESHOLD, color="#e74c3c", linestyle="--",
        label=f"Umbral {PCA_VARIANCE_THRESHOLD*100:.0f}%")
ax.axvline(x=n_components_95, color="#2ecc71", linestyle="--",
        label=f"{n_components_95} comp.")
ax.set_xlabel("Componente")
ax.set_ylabel("Varianza Explicada")
ax.set_title("Scree Plot — PCA Spark MLlib")
ax.legend()
ax.set_xlim(0, min(50, len(explained_variance)))
plt.tight_layout()
save_figure(fig, "32_pca_scree_plot_formal")

🔍 PCA con Spark MLlib...


26/05/04 22:58:30 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/04 22:58:36 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


  45 features → 27 componentes (95% varianza)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/32_pca_scree_plot_formal.png


---
## 4.7 — ANOVA F-test (Selección Univariante)

**Nota:** El `ChiSqSelector` de Spark requiere variables categóricas (discretas).
Nuestras features son continuas (>10.000 valores únicos por columna), por lo que
Chi² no es aplicable y lanza error.

Se usa en su lugar **ANOVA F-test** (`f_classif` de sklearn), que es el equivalente
para variables continuas: mide si la media de cada feature difiere significativamente
entre las clases del target.

> **Ref:** Zouhri et al. (2024) evalúan múltiples filtros univariantes para
> CSE-CIC-IDS 2018, incluyendo ANOVA como alternativa a Chi² para features continuas.

In [9]:
print("🔍 ANOVA F-test (selección univariante para features continuas)...")

try:
    from sklearn.feature_selection import f_classif

    SAMPLE_ANOVA = min(100_000, initial_rows)
    df_anova = (
        df.select(feature_cols_reduced + ["label_index"])
        .sample(False, SAMPLE_ANOVA / initial_rows, seed=SEED)
        .toPandas()
    )

    X_anova = df_anova[feature_cols_reduced].apply(pd.to_numeric, errors="coerce").fillna(0)
    y_anova = df_anova["label_index"].astype(int)

    f_scores, p_values = f_classif(X_anova, y_anova)

    anova_df = (
        pd.DataFrame({
            "feature": feature_cols_reduced,
            "F_score": f_scores,
            "p_value": p_values
        })
        .sort_values("F_score", ascending=False)
    )

    N_ANOVA = min(30, len(feature_cols_reduced))
    anova_top = anova_df.head(N_ANOVA)

    print(f"\n  Top {N_ANOVA} features por ANOVA F-score:")
    for i, row in enumerate(anova_top.itertuples()):
        print(f"	{i+1:>2}. {row.feature:<40} F={row.F_score:>12,.1f}  p={row.p_value:.2e}")

    anova_names = list(anova_top["feature"])

except ImportError:
    print("  ⚠️ sklearn no disponible — se omite ANOVA F-test")
    anova_names = []
    anova_df = pd.DataFrame()
    N_ANOVA = 0

🔍 ANOVA F-test (selección univariante para features continuas)...



  Top 30 features por ANOVA F-score:
	 1. Tot Fwd Pkts                             F=    12,578.4  p=0.00e+00
	 2. Fwd Seg Size Min                         F=     2,241.1  p=0.00e+00
	 3. Init Fwd Win Byts                        F=     1,845.6  p=0.00e+00
	 4. day_of_week                              F=     1,508.8  p=0.00e+00
	 5. hour                                     F=     1,293.7  p=0.00e+00
	 6. Bwd Pkts/s                               F=     1,147.4  p=0.00e+00
	 7. ACK Flag Cnt                             F=       891.0  p=0.00e+00
	 8. Bwd IAT Mean                             F=       738.4  p=0.00e+00
	 9. Bwd IAT Min                              F=       610.8  p=0.00e+00
	10. Fwd Pkt Len Mean                         F=       510.3  p=0.00e+00
	11. Protocol                                 F=       436.2  p=0.00e+00
	12. Fwd Pkt Len Max                          F=       400.9  p=0.00e+00
	13. RST Flag Cnt                             F=       378.7  p=0.00e+00
	14. Bwd Pkt 

---
## 4.8 — VIF (Variance Inflation Factor)

**Nota:** VIF detecta multicolinealidad residual que la correlación de Pearson
podría no capturar (relaciones lineales entre 3+ variables).
Se ejecuta sobre una muestra en pandas por eficiencia.

In [10]:
print("🔍 Calculando VIF...")

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    SAMPLE_VIF = min(50_000, initial_rows)
    df_vif = (
        df.select(feature_cols_reduced)
        .sample(False, SAMPLE_VIF / initial_rows, seed=SEED)
        .toPandas()
    )
    df_vif = df_vif.apply(pd.to_numeric, errors="coerce").dropna()

    vif_data = []
    for i, col_name in enumerate(feature_cols_reduced):
        try:
            vif_val = variance_inflation_factor(df_vif.values, i)
            vif_data.append({"feature": col_name, "VIF": round(vif_val, 2)})
        except Exception:
            vif_data.append({"feature": col_name, "VIF": float("inf")})

    vif_df = pd.DataFrame(vif_data).sort_values("VIF", ascending=False)
    high_vif = vif_df[vif_df["VIF"] > VIF_THRESHOLD]

    print(f"  Features con VIF > {VIF_THRESHOLD}: {len(high_vif)}")
    if len(high_vif) > 0:
        print(high_vif.head(15).to_string(index=False))

except ImportError:
    print("  ⚠️ statsmodels no disponible — se omite VIF")
    high_vif = pd.DataFrame()
    vif_df = pd.DataFrame()

🔍 Calculando VIF...


  Features con VIF > 10: 26
         feature     VIF
     Active Mean 1822.28
      Active Min 1706.72
      Active Std  535.74
      Active Max  176.61
    Flow IAT Max  126.96
Bwd Pkt Len Mean   95.58
   Flow IAT Mean   78.71
     Bwd IAT Min   65.60
    Bwd IAT Mean   64.20
     Bwd IAT Max   63.33
    Pkt Len Mean   62.20
Fwd Seg Size Min   44.42
     Bwd IAT Std   42.36
   Flow Duration   33.22
     Pkt Len Var   32.90


---
## 4.9 — Tabla Comparativa de Métodos

Resumen de los resultados de cada técnica de selección aplicada.
La decisión final se basa en la **correlación** como filtro principal,
complementado por la información de los otros métodos.

In [11]:
print("\n" + "=" * 70)
print("📋 TABLA COMPARATIVA — SELECCIÓN DE FEATURES")
print("=" * 70)
print(f"  Features originales (NB3):   	{len(feature_cols)}")
print(f"  Tras correlación (> {CORRELATION_THRESHOLD}): 	{len(feature_cols_reduced)}")
print(f"  Componentes PCA (95% var.):  	{n_components_95}")
print(f"  ANOVA F-test top:            	{N_ANOVA}")
if len(high_vif) > 0:
    print(f"  Features con VIF > {VIF_THRESHOLD}:      	{len(high_vif)}")
print("=" * 70)

# Consenso: features que aparecen en RF top 20 Y ANOVA top N
rf_top20 = set(importance_df.head(20)["feature"])
anova_top_set = set(anova_names) if anova_names else set()
consensus = rf_top20 & anova_top_set
print(f"\n  Features en RF top 20 ∩ ANOVA top {N_ANOVA}: {len(consensus)}")
for f in sorted(consensus):
    print(f"	· {f}")


📋 TABLA COMPARATIVA — SELECCIÓN DE FEATURES
  Features originales (NB3):   	72
  Tras correlación (> 0.95): 	45
  Componentes PCA (95% var.):  	27
  ANOVA F-test top:            	30
  Features con VIF > 10:      	26

  Features en RF top 20 ∩ ANOVA top 30: 17
	· Bwd Pkt Len Max
	· Bwd Pkt Len Mean
	· Bwd Pkts/s
	· Dst Port
	· Flow Duration
	· Flow IAT Max
	· Flow IAT Mean
	· Flow Pkts/s
	· Fwd Pkt Len Max
	· Fwd Pkt Len Mean
	· Fwd Seg Size Min
	· Idle Mean
	· Init Fwd Win Byts
	· Pkt Len Mean
	· Tot Fwd Pkts
	· day_of_week
	· hour


---
## 4.10 — Decisión Final de Features

**Criterio adoptado:** Se conserva el set filtrado por correlación
(`feature_cols_reduced`) como conjunto final. Razones:

1. La correlación elimina redundancia real (features casi idénticas).
2. Los otros métodos (RF, PCA, ANOVA, VIF) son informativos y se
	discuten en la memoria del TFG, pero no eliminan features adicionales
	en este punto.
3. Es el enfoque más conservador: mantener todas las features
	no-redundantes y dejar que el modelo aprenda la importancia.

> **Ref:** Göcs & Johanyák (2023) siguen este mismo enfoque:
> correlación primero, luego múltiples métodos de ranking para validar.

In [12]:
# Eliminar del DataFrame las features descartadas por correlación
cols_to_keep = ["Label", "label_index"] + feature_cols_reduced
df_final = df.select(cols_to_keep)

print(f"📊 Dataset final para modelado:")
print(f"  Filas:	{df_final.count():,}")
print(f"  Columnas: {len(df_final.columns)}")
print(f"  Features: {len(feature_cols_reduced)}")

📊 Dataset final para modelado:
  Filas:	15,705,438
  Columnas: 47
  Features: 45


---
## 4.11 — Guardado Final

Se guarda el dataset con:
- Features individuales seleccionadas (sin vector, sin escalado)
- `Label` (string) y `label_index` (numérico)

**NB5** se encargará de:
1. Train/test split (stratified)
2. Balanceo de clases (SOLO sobre train)
3. VectorAssembler
4. StandardScaler (fit SOLO en train)
5. Entrenamiento de modelos

In [13]:
final_path = os.path.join(DATA_FINAL_PATH, "dataset_final")

print(f"💾 Guardando en {final_path}...")
df_final.write.mode("overwrite").parquet(final_path)

# Verificación
df_check = spark.read.parquet(final_path)

from pathlib import Path

print("\n" + "=" * 70)
print("📋 RESUMEN FINAL — NB4 FEATURE ENGINEERING Y SELECCIÓN")
print("=" * 70)
print(f"  Filas:                	{df_check.count():,}")
print(f"  Features originales:  	{len(feature_cols)}")
print(f"  Features seleccionadas:   {len(feature_cols_reduced)}")
print(f"  Features eliminadas:  	{len(feature_cols) - len(feature_cols_reduced)}")
print(f"  Nuevas features:      	hour, day_of_week")
print(f"  Componentes PCA (ref.):   {n_components_95}")
if os.path.exists(final_path):
    s = sum(f.stat().st_size for f in Path(final_path).rglob("*") if f.is_file())
    print(f"  Tamaño en disco:      	{s / (1024**3):.2f} GB")
print(f"  Ruta salida:          	{final_path}")
print("=" * 70)
print("\n➡️ Siguiente: NB5 — Train/Test Split, Balanceo y Entrenamiento")
print("   · Balanceo SOLO sobre train (evitar data leakage)")
print("   · StandardScaler fit SOLO en train")

💾 Guardando en /home/hpc/22231088student/tfg_ids/data/final/dataset_final...



📋 RESUMEN FINAL — NB4 FEATURE ENGINEERING Y SELECCIÓN
  Filas:                	15,705,438
  Features originales:  	72
  Features seleccionadas:   45
  Features eliminadas:  	27
  Nuevas features:      	hour, day_of_week
  Componentes PCA (ref.):   27
  Tamaño en disco:      	1.28 GB
  Ruta salida:          	/home/hpc/22231088student/tfg_ids/data/final/dataset_final

➡️ Siguiente: NB5 — Train/Test Split, Balanceo y Entrenamiento
   · Balanceo SOLO sobre train (evitar data leakage)
   · StandardScaler fit SOLO en train


In [14]:
spark.stop()